## Step 1: Load the API key

This notebook reads the ExchangeRate-API key from the environment. The key is required for the live currency conversion request.

In [1]:
import os
%pip install python-dotenv
from dotenv import load_dotenv
load_dotenv()

api = os.environ["API"]
print("API key loaded:", bool(api))

Note: you may need to restart the kernel to use updated packages.
API key loaded: True


## Step 2: Import required libraries

We import the local LLM wrapper, tool decorators, message types, HTTP library, and the injected tool argument helper needed for tool execution.

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from typing import Annotated
from langchain_core.tools import InjectedToolArg

## Step 3: Define the conversion tools

Here we create two LangChain tools: one to fetch the current exchange rate from the API and one to multiply an amount by that rate. The second tool uses an injected argument so the model can propose a conversion call while the notebook fills in the live rate.

In [3]:
@tool
def convert_currency(base_currency: str, target_currency: str) -> float:
    """This function fetches the exchange rate between two currencies. it takes two parameters: base_currency and target_currency. It returns the exchange rate as a float."""
    url = f"https://v6.exchangerate-api.com/v6/{api}/pair/{base_currency}/{target_currency}"
    response = requests.get(url)
    return response.json()

@tool
def convert(base_currency_value: float, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """Given the the conversion rate this function is to convert the currency value based on the conversion rate.
    It takes two parameters: base_currency_value and conversion_rate.
    It returns the converted amount."""
    return base_currency_value * conversion_rate

# @tool 
# def convert(base_currency_value: float, conversion_rate: float) -> float:
#     """This function is to convert the currency value based on the conversion rate.
#     It takes two parameters: base_currency_value and conversion_rate.
#     It returns the converted amount."""
#     return base_currency_value * conversion_rate



In [4]:
# testing API
url = f"https://v6.exchangerate-api.com/v6/{api}/pair/USD/INR"
response = requests.get(url)
response.json()

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1791849601,
 'time_last_update_utc': 'Tue, 13 Oct 2026 00:00:01 +0000',
 'time_next_update_unix': 1791936001,
 'time_next_update_utc': 'Wed, 14 Oct 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.9069}

In [5]:
convert_currency.invoke({'base_currency': 'USD', 'target_currency': 'INR'})


{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1791849601,
 'time_last_update_utc': 'Tue, 13 Oct 2026 00:00:01 +0000',
 'time_next_update_unix': 1791936001,
 'time_next_update_utc': 'Wed, 14 Oct 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.9069}

In [6]:
convert.invoke({'base_currency_value': 100, 'conversion_rate': 82.5})

8250.0

## Step 4: Bind tools to the local LLM

We create an Ollama chat model and bind both conversion tools to it. This allows the model to decide when to call the exchange-rate tool and the conversion tool in response to a user request.

In [7]:
llm = ChatOllama(model="qwen2.5:3b", temperature=0)

In [8]:
llm_tools = llm.bind_tools([convert_currency, convert])

## Step 5: Create the user prompt

This message asks the model to fetch the USD-to-INR rate and then convert 100 USD. The model will decide which tool calls it needs based on this request.

In [9]:
messages = [HumanMessage("What is the conversion rate of AUD to INR? Can you convert 10 AUD to INR?") ]

In [10]:
messages 

[HumanMessage(content='What is the conversion rate of AUD to INR? Can you convert 10 AUD to INR?', additional_kwargs={}, response_metadata={})]

## Step 6: Ask the model for tool calls

The model inspects the user request and decides which tool or tools it needs to answer correctly. Here we inspect the returned `tool_calls` structure.

In [11]:
llm_tools.invoke(messages)

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-10-13T14:45:50.513422Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6820427708, 'load_duration': 18827500, 'prompt_eval_count': 276, 'prompt_eval_duration': 47544000, 'eval_count': 82, 'eval_duration': 6737178000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01a135b9-fc88-7751-b939-f00bdaee5453-0', tool_calls=[{'name': 'convert_currency', 'args': {'base_currency': 'AUD', 'target_currency': 'INR'}, 'id': '1db54f30-a18c-421f-a1e8-31cb7ae6ffc3', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10, 'conversion_rate': 73.5}, 'id': '0e1230bf-421b-46b3-8a83-0558d5a9798e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 276, 'output_tokens': 82, 'total_tokens': 358})

In [12]:
llm_tools.invoke(messages).tool_calls

[{'name': 'convert_currency',
  'args': {'base_currency': 'AUD', 'target_currency': 'INR'},
  'id': '0ccc7ced-02ce-4e99-b646-7a5ca662d522',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10, 'conversion_rate': 73.5},
  'id': 'b353bd5c-ceeb-4a14-8810-c092b5212b83',
  'type': 'tool_call'}]

## Step 7: Execute the model's selected tools

Once the tool call is identified, we manually execute the matching tool. The exchange-rate tool returns the live rate, and then we inject that rate into the conversion tool before running it.

In [13]:
ai_messages = llm_tools.invoke(messages)

In [14]:
messages.append(ai_messages)

In [15]:
import json

for tool_call in ai_messages.tool_calls:
    if tool_call["name"] == "convert_currency":
        tool_message1 = convert_currency.invoke(tool_call)
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        messages.append(tool_message1)
    if tool_call["name"] == "convert":
        tool_call["args"]["conversion_rate"] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

In [16]:
messages

[HumanMessage(content='What is the conversion rate of AUD to INR? Can you convert 10 AUD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-10-13T14:46:02.118405Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5630321750, 'load_duration': 3925042, 'prompt_eval_count': 276, 'prompt_eval_duration': 62492000, 'eval_count': 82, 'eval_duration': 5560261000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01a135ba-2e85-78d3-81ec-03d209b9ba3c-0', tool_calls=[{'name': 'convert_currency', 'args': {'base_currency': 'AUD', 'target_currency': 'INR'}, 'id': '3af5d2ee-8f41-450c-9fe0-33ced6916af7', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10, 'conversion_rate': 66.7763}, 'id': '13baaf24-1284-4163-9e70-28245f924dc1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 276, 'output_token

## Step 8: Get the final response from the model

Now that the tool results are added to the message history, we ask the model again to produce the final answer. The model combines the tool outputs and writes the final user-facing response.

In [17]:
response = llm_tools.invoke(messages)
response

AIMessage(content='The conversion rate from AUD to INR is 66.7763. After converting 10 AUD to INR, you would get approximately 667.763 INR. Please note that the exact amount might vary slightly due to rounding or other factors.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-10-13T14:46:08.570716Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6018122250, 'load_duration': 8111333, 'prompt_eval_count': 534, 'prompt_eval_duration': 1132018000, 'eval_count': 60, 'eval_duration': 4821648000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01a135ba-4637-75d3-89d8-d91363bbea8f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 534, 'output_tokens': 60, 'total_tokens': 594})

In [18]:
response.content

'The conversion rate from AUD to INR is 66.7763. After converting 10 AUD to INR, you would get approximately 667.763 INR. Please note that the exact amount might vary slightly due to rounding or other factors.'

## Summary

This notebook demonstrates the full tool-calling workflow in LangChain. The LLM decides which tools to call, we execute those tools, inject the results into the conversation, and then ask the model to answer the original user request using the live tool output. This pattern is the foundation of agentic AI systems.

lets write complete in a single function which will take the base currency, target currency and the amount to be converted as input and return the converted amount as output.

In [ ]:
get_rate(input("Enter your query: "))
def get_rate(messages):
    ai_messages = llm_tools.invoke(messages)
    messages.append(ai_messages)
    for tool_call in ai_messages.tool_calls:
        if tool_call["name"] == "convert_currency":
            tool_message1 = convert_currency.invoke(tool_call)
            conversion_rate = json.loads(tool_message1.content)['conversion_rate']
            messages.append(tool_message1)
        if tool_call["name"] == "convert":
            tool_call["args"]["conversion_rate"] = conversion_rate
            tool_message2 = convert.invoke(tool_call)
            messages.append(tool_message2)
    return llm_tools.invoke(messages).content